# RQ2 — Prompt Injection Walkthrough

**Purpose:** This notebook provides a step-by-step walkthrough of how *ESRS context injection*
(referred to as "prompt injection" in the thesis) is implemented in the RQ2 evaluation pipeline.
It shows exactly what the LLM sees in each experimental condition and demonstrates the difference
with a concrete example.

**Research Question 2:** *To what extent does full-standard ESRS context injection improve
evidence-grounded compliance assessment?*

**Experimental conditions:**

| Condition | Label | What the LLM receives |
|-----------|-------|----------------------|
| **full_system** | With ESRS context | System prompt + Question + **Injected ESRS requirements** + Retrieved evidence |
| **no_esrs_context** | Without ESRS context | System prompt + Question + *(empty ESRS block)* + Retrieved evidence |

The only variable that changes between conditions is whether the ESRS regulatory standard
text is injected into the prompt. Everything else — the retrieval pipeline, the system prompt,
the company report evidence, and the output schema — remains identical.

## 1. Setup and Imports

In [1]:
import sys, os, json, textwrap
from pathlib import Path
from IPython.display import display, Markdown, HTML

# Ensure the project root is importable
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.config.settings import (
    GOLDEN_STANDARD_PATH, ESRS_JSON_DIR, RQ2_RESULTS_DIR, BEST_PIPELINE_PATH,
)
from rag.data.loaders import load_generation_samples
from rag.generation.esrs_context import (
    load_esrs_standards, extract_esrs_context, build_disclosure_context,
)
from rag.generation.prompts import build_answer_messages, format_retrieved_chunks
from rag.generation.generator import AnswerGenerator, generate_answer
from rag.types import GenerationSample, RetrievedEvidenceChunk

print(f"Project root : {PROJECT_ROOT}")
print(f"ESRS JSON dir: {ESRS_JSON_DIR}")
print(f"RQ2 cache dir: {RQ2_RESULTS_DIR / 'nb_cache'}")

Project root : /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot
ESRS JSON dir: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/ESRS_Standards_JSON
RQ2 cache dir: /Users/sinahaghgoo/Library/CloudStorage/OneDrive-Persönlich/Master Thesis/Thesis Coding Part/ESRS-Regulation-Bot/data/artifacts/evaluation_results/rq2/nb_cache


## 2. Architecture Overview — Where Prompt Injection Happens

The RQ2 pipeline follows a standard RAG (Retrieval-Augmented Generation) pattern with one
experimental twist: the **ESRS regulatory context** can be injected or withheld.

```
┌──────────────────────────────────────────────────────────────────────┐
│                        RQ2 Generation Pipeline                       │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────────┐    ┌──────────────────┐    ┌────────────────────┐  │
│  │ Golden       │───▶│ Best RQ1         │───▶│ Retrieved Evidence │  │
│  │ Standard     │    │ Retrieval        │    │ Chunks (top-15)    │  │
│  │ (question +  │    │ Pipeline         │    │                    │  │
│  │  company)    │    │                  │    │ (identical for     │  │
│  └─────────────┘    └──────────────────┘    │  both conditions)  │  │
│                                              └─────────┬──────────┘  │
│                                                        │             │
│  ┌─────────────────────────────────┐                   │             │
│  │ ESRS Standards JSON             │                   │             │
│  │ (e.g. ESRS_S2.json)            │                   │             │
│  │                                 │                   │             │
│  │ ┌─ full_system ──────────────┐  │                   │             │
│  │ │  Flatten & serialize full  │──┼───┐               │             │
│  │ │  regulatory text           │  │   │               │             │
│  │ └───────────────────────────┘  │   │               │             │
│  │ ┌─ no_esrs_context ─────────┐  │   │               │             │
│  │ │  Empty string ""          │──┼───┤               │             │
│  │ └───────────────────────────┘  │   │               │             │
│  └─────────────────────────────────┘   │               │             │
│                                        ▼               ▼             │
│                              ┌──────────────────────────────┐       │
│                              │      build_answer_messages()  │       │
│                              │  ┌──────────────────────────┐ │       │
│                              │  │ SYSTEM: evaluation rubric │ │       │
│                              │  │ USER:   question          │ │       │
│                              │  │       + ESRS context ◄────┼─┼─ THE INJECTION POINT
│                              │  │       + evidence chunks   │ │       │
│                              │  └──────────────────────────┘ │       │
│                              └──────────────┬───────────────┘       │
│                                             │                        │
│                                             ▼                        │
│                              ┌──────────────────────────────┐       │
│                              │  OpenAI API (gpt-4 mini)          │       │
│                              │  + JSON schema enforcement    │       │
│                              └──────────────┬───────────────┘       │
│                                             │                        │
│                                             ▼                        │
│                              ┌──────────────────────────────┐       │
│                              │  Structured JSON response:    │       │
│                              │  • compliance_level           │       │
│                              │  • compliance_rationale       │       │
│                              │  • key_findings               │       │
│                              │  • disclosure_gaps            │       │
│                              │  • cited_pages                │       │
│                              └──────────────────────────────┘       │
└──────────────────────────────────────────────────────────────────────┘
```

The key source files involved:

| File | Role |
|------|------|
| `rag/generation/esrs_context.py` | Loads and serializes ESRS JSON into a prompt-ready text block |
| `rag/generation/prompts.py` | Assembles the full system + user message with the injection point |
| `rag/generation/generator.py` | Calls the OpenAI API with strict JSON schema enforcement |

## 3. Concrete Example — Aker BP × ESRS S2 (Workers in the Value Chain)

We use a single evaluation sample to walk through every component:

- **Company:** Aker BP
- **Question (Q9):** *"What policies and engagement processes does the undertaking have in place to identify and address negative impacts on workers in its value chain, including regarding forced labor and child labor?"*
- **ESRS Standard:** ESRS S2 — Workers in the Value Chain
- **Disclosure IDs:** S2-1, S2-2
- **Gold-standard fulfillment level:** High

### 3.1 Load the Gold-Standard Sample

In [2]:
# Load all 204 generation samples and pick our example
all_samples = load_generation_samples()

# Filter for Aker BP × ESRS S2 (Q9)
sample = next(s for s in all_samples if s.company == "Aker BP" and s.esrs_standard == "ESRS S2")

print(f"Question ID       : {sample.question_id}")
print(f"Company           : {sample.company}")
print(f"ESRS Standard     : {sample.esrs_standard}")
print(f"Disclosure IDs    : {sample.esrs_disclosure_id}")
print(f"Gold fulfillment  : {sample.fulfillment_level}")
print(f"# Gold evidences  : {len(sample.evidences)}")
print(f"\nQuestion:\n{sample.question}")

Question ID       : Q9
Company           : Aker BP
ESRS Standard     : ESRS S2
Disclosure IDs    : S2-1, S2-2
Gold fulfillment  : High
# Gold evidences  : 3

Question:
What policies and engagement processes does the undertaking have in place to identify and address negative impacts on workers in its value chain, including regarding forced labor and child labor?


### 3.2 Inspect the Gold-Standard Evidence Passages

These are the expert-annotated passages from Aker BP's sustainability report that the retrieval
pipeline should find. They serve as both the ground truth for retrieval evaluation (RQ1) and
the evidence base the LLM will use for generation (RQ2).

In [3]:
for i, ev in enumerate(sample.evidences, 1):
    print(f"{'='*80}")
    print(f"Evidence {i} | Page {ev.page_number} | Section: {ev.source_section_header}")
    print(f"{'='*80}")
    print(textwrap.fill(ev.text[:500], width=100))
    if len(ev.text) > 500:
        print(f"  ... [{len(ev.text)} chars total]")
    print()

Evidence 1 | Page 120 | Section: 8.1 POLICIES AND PROCEDURES
Our code of conduct and human rights policy addresses material topics such as working conditions,
human rights (including child, forced and compulsory labour), violence and harassment, and health
and safety.

Evidence 2 | Page 120 | Section: 8.1 POLICIES AND PROCEDURES
Before establishing or amending any business relationship, we must follow Aker BP's business partner
integrity procedure to assess legal and ethical risks and include contractual obligations on ethics
and compliance where applicable. This process pays special attention to material topics for value
chain workers. To clearly communicate our expectations to our business partners and monitor their
compliance where necessary. Requirements are specified in contracts and Aker BP's supplier
declaration,
  ... [584 chars total]

Evidence 3 | Page 121 | Section: 8.2.2 Human rights audits
We conduct audits of a selection of our suppliers and business partners to verify th

## 4. Component 1 — The ESRS Context (The "Injection")

This is the core of the experiment. The ESRS standard text is loaded from a structured JSON file
(`data/ESRS_Standards_JSON/ESRS_S2.json`), flattened into a hierarchical text representation,
and filtered to the relevant disclosure IDs (S2-1, S2-2).

The function `build_disclosure_context()` in `rag/generation/esrs_context.py` handles this:
1. It parses the nested JSON tree of the ESRS standard
2. It filters to subtrees matching the disclosure IDs (e.g., S2-1, S2-2)
3. It serializes each paragraph as `[Paragraph X / Section > Heading] text`

This text block is what gets **injected** into the prompt in the `full_system` condition.
In the `no_esrs_context` condition, this is replaced with an empty string.

In [4]:
# Load all ESRS standard JSON files
standards = load_esrs_standards()
print(f"Loaded {len(standards)} ESRS standards: {list(standards.keys())}")

# Extract the ESRS context for our specific question
esrs_context = extract_esrs_context(
    esrs_standard=sample.esrs_standard,
    standards=standards,
    question=sample.question,
    esrs_disclosure_id=sample.esrs_disclosure_id,
)

print(f"\nESRS context length: {len(esrs_context):,} characters")
print(f"\n{'='*80}")
print("FULL ESRS CONTEXT (first 3000 chars):")
print(f"{'='*80}")
print(esrs_context[:3000])
if len(esrs_context) > 3000:
    print(f"\n... [{len(esrs_context):,} chars total — truncated for display]")

Loaded 12 ESRS standards: ['ESRS 1', 'ESRS 2', 'ESRS E1', 'ESRS E2', 'ESRS E3', 'ESRS E4', 'ESRS E5', 'ESRS G1', 'ESRS S1', 'ESRS S2', 'ESRS S3', 'ESRS S4']

ESRS context length: 19,566 characters

FULL ESRS CONTEXT (first 3000 chars):
Question: What policies and engagement processes does the undertaking have in place to identify and address negative impacts on workers in its value chain, including regarding forced labor and child labor?
ESRS standard: ESRS S2
Mapped disclosure ID: S2-1, S2-2

Relevant ESRS disclosure requirements:
[Paragraph 14 / ESRS S2 > WORKERS IN THE VALUE CHAIN > WORKERS IN THE VALUE CHAIN > Disclosure Requirements > Impact, risk and opportunity management > Disclosure Requirement S2-1 – Policies related to value chain workers] The undertaking shall describe its policies adopted to manage its material impacts on value chain workers, as well as associated material risks and opportunities.
[Paragraph 15 / ESRS S2 > WORKERS IN THE VALUE CHAIN > WORKERS IN THE VALUE 

### Comparison: What Each Condition Sees

| | `full_system` | `no_esrs_context` |
|---|---|---|
| **ESRS context** | ~14,000–25,000 chars of regulatory text | `""` (empty string) |
| **Everything else** | Identical | Identical |

The hypothesis is that injecting this regulatory text helps the LLM better interpret what
constitutes adequate disclosure fulfillment. The experiment tests whether this actually improves
compliance assessment accuracy and rationale quality.

## 5. Component 2 — The System Prompt (Identical for Both Conditions)

The system prompt defines the LLM's role and the evaluation rubric. It is **identical** across
both conditions — this is critical for experimental control.

Key elements of the system prompt:
- **Role definition:** The LLM evaluates ESRS disclosure fulfillment
- **Compliance rubric:** High / Medium / Low with clear definitions
- **Grounding constraint:** All claims must be grounded in retrieved evidence
- **ESRS context instruction:** "Use the ESRS requirements block only as interpretive context"
- **Output schema:** Specifies the exact JSON keys the model must return

In [5]:
# Build mock evidence chunks from the gold standard to demonstrate prompt construction
mock_chunks = [
    RetrievedEvidenceChunk(
        chunk_id=f"gold_{i}",
        text=ev.text,
        score=1.0,
        rank=i + 1,
        page_start=int(ev.page_number) if ev.page_number.isdigit() else None,
        page_end=int(ev.page_number) if ev.page_number.isdigit() else None,
        header_hierarchy=[ev.source_section_header] if ev.source_section_header else [],
    )
    for i, ev in enumerate(sample.evidences)
]

# Build the full prompt messages for the full_system condition
messages_with_context = build_answer_messages(
    sample=sample,
    retrieved_chunks=mock_chunks,
    esrs_context=esrs_context,
)

# Display the system prompt
print("=" * 80)
print("SYSTEM PROMPT (shared by both conditions)")
print("=" * 80)
print(textwrap.fill(messages_with_context[0]["content"], width=100))

SYSTEM PROMPT (shared by both conditions)
You evaluate ESRS disclosure fulfillment for a company's sustainability report. Important: the
compliance level refers to disclosure fulfillment, not to whether the company's sustainability
performance is good or bad. Evaluate fulfillment at the level of the specific research question, not
as an exhaustive ESRS checklist unless the question explicitly asks for a full checklist. Ground
every claim in the retrieved report evidence. Use the ESRS requirements block only as interpretive
context for the question. Do not downgrade solely because the disclosure is not presented in a
single table, does not use exact ESRS wording, or lacks data-point-level tagging, if the retrieved
evidence still clearly and substantially answers the question. If the company clearly discloses that
something is absent, not applicable, or negative, that can still support High disclosure fulfillment
when it directly answers the question. If a required detail is genuinely mi

## 6. Component 3 — The User Message (Side-by-Side Comparison)

This is where the injection happens. The user message is constructed by `build_answer_messages()`
in `rag/generation/prompts.py`. Its structure is:

```
Question ID: {question_id}
Company: {company}
ESRS standard: {esrs_standard}
Disclosure ID: {disclosure_id}
Question: {question}

Injected ESRS requirements:
{esrs_context}              ← THIS IS THE INJECTION POINT

Retrieved company report evidence:
{evidence_block}

Write a faithful direct answer, assign the disclosure fulfillment level, ...
```

Let's see the actual user messages for both conditions side by side.

In [6]:
# Build messages for BOTH conditions
messages_full = build_answer_messages(
    sample=sample, retrieved_chunks=mock_chunks, esrs_context=esrs_context,
)
messages_no_ctx = build_answer_messages(
    sample=sample, retrieved_chunks=mock_chunks, esrs_context="",
)

user_full = messages_full[1]["content"]
user_no_ctx = messages_no_ctx[1]["content"]

print(f"User message length WITH ESRS context   : {len(user_full):>8,} chars")
print(f"User message length WITHOUT ESRS context : {len(user_no_ctx):>8,} chars")
print(f"Difference (= injected ESRS text)        : {len(user_full) - len(user_no_ctx):>8,} chars")

User message length WITH ESRS context   :   21,990 chars
User message length WITHOUT ESRS context :    2,424 chars
Difference (= injected ESRS text)        :   19,566 chars


### 6.1 User Message — `full_system` (WITH ESRS Context Injection)

In [7]:
# Show the full user message with ESRS context (truncated for readability)
lines = user_full.split("\n")

# Find the key sections
print("─" * 80)
print("USER MESSAGE — full_system condition")
print("─" * 80)

# Print first lines (question metadata)
for line in lines[:6]:
    print(line)

# Show the ESRS injection section (truncated)
print("\nInjected ESRS requirements:")
esrs_start = user_full.find("Injected ESRS requirements:\n") + len("Injected ESRS requirements:\n")
esrs_end = user_full.find("\nRetrieved company report evidence:")
esrs_block = user_full[esrs_start:esrs_end]
esrs_lines = esrs_block.strip().split("\n")
print(f"  ┌── ESRS CONTEXT BLOCK ({len(esrs_lines)} lines, {len(esrs_block):,} chars) ──")
for line in esrs_lines[:10]:
    print(f"  │ {line[:120]}")
print(f"  │ ... [{len(esrs_lines) - 10} more lines]")
print(f"  └── END ESRS CONTEXT ──")

# Show the evidence section header
print("\nRetrieved company report evidence:")
evidence_start = user_full.find("Retrieved company report evidence:\n") + len("Retrieved company report evidence:\n")
evidence_block = user_full[evidence_start:]
evidence_lines = evidence_block.strip().split("\n")
for line in evidence_lines[:8]:
    print(f"  {line[:120]}")
print(f"  ... [{len(evidence_lines) - 8} more lines of evidence]")

────────────────────────────────────────────────────────────────────────────────
USER MESSAGE — full_system condition
────────────────────────────────────────────────────────────────────────────────
Question ID: Q9
Company: Aker BP
ESRS standard: ESRS S2
Disclosure ID: S2-1, S2-2
Question: What policies and engagement processes does the undertaking have in place to identify and address negative impacts on workers in its value chain, including regarding forced labor and child labor?


Injected ESRS requirements:
  ┌── ESRS CONTEXT BLOCK (27 lines, 19,567 chars) ──
  │ Question: What policies and engagement processes does the undertaking have in place to identify and address negative imp
  │ ESRS standard: ESRS S2
  │ Mapped disclosure ID: S2-1, S2-2
  │ 
  │ Relevant ESRS disclosure requirements:
  │ [Paragraph 14 / ESRS S2 > WORKERS IN THE VALUE CHAIN > WORKERS IN THE VALUE CHAIN > Disclosure Requirements > Impact, ri
  │ [Paragraph 15 / ESRS S2 > WORKERS IN THE VALUE CHAIN > WORKERS I

### 6.2 User Message — `no_esrs_context` (WITHOUT ESRS Context)

In [8]:
# Show the user message WITHOUT ESRS context
lines_no_ctx = user_no_ctx.split("\n")

print("─" * 80)
print("USER MESSAGE — no_esrs_context condition")
print("─" * 80)

# Print first lines (question metadata)
for line in lines_no_ctx[:6]:
    print(line)

# Show the empty ESRS section
print("\nInjected ESRS requirements:")
esrs_start_no = user_no_ctx.find("Injected ESRS requirements:\n") + len("Injected ESRS requirements:\n")
esrs_end_no = user_no_ctx.find("\nRetrieved company report evidence:")
esrs_block_no = user_no_ctx[esrs_start_no:esrs_end_no]
print(f"  ┌── ESRS CONTEXT BLOCK ──")
print(f"  │ {repr(esrs_block_no.strip())}")
print(f"  └── (EMPTY — no regulatory text injected) ──")

# Show the evidence section header (same as before)
print("\nRetrieved company report evidence:")
evidence_start_no = user_no_ctx.find("Retrieved company report evidence:\n") + len("Retrieved company report evidence:\n")
evidence_block_no = user_no_ctx[evidence_start_no:]
evidence_lines_no = evidence_block_no.strip().split("\n")
for line in evidence_lines_no[:8]:
    print(f"  {line[:120]}")
print(f"  ... [same {len(evidence_lines_no)} lines of evidence as full_system]")

────────────────────────────────────────────────────────────────────────────────
USER MESSAGE — no_esrs_context condition
────────────────────────────────────────────────────────────────────────────────
Question ID: Q9
Company: Aker BP
ESRS standard: ESRS S2
Disclosure ID: S2-1, S2-2
Question: What policies and engagement processes does the undertaking have in place to identify and address negative impacts on workers in its value chain, including regarding forced labor and child labor?


Injected ESRS requirements:
  ┌── ESRS CONTEXT BLOCK ──
  │ ''
  └── (EMPTY — no regulatory text injected) ──

Retrieved company report evidence:
  [Evidence 1]
  Pages: 120
  Section: 8.1 POLICIES AND PROCEDURES
  Text:
  Our code of conduct and human rights policy addresses material topics such as working conditions, human rights (includin
  
  [Evidence 2]
  Pages: 120
  ... [same 19 lines of evidence as full_system]


## 7. Component 4 — The JSON Schema Enforcement

The OpenAI API call uses **strict JSON schema mode** to constrain the model's output to a
well-defined structure. This ensures that both conditions produce responses in the same format,
making them directly comparable.

The schema is defined in `rag/generation/generator.py` and enforced via the OpenAI Responses API's
`text.format.json_schema` parameter with `strict: True`.

In [9]:
# The JSON schema enforced on the LLM output
output_schema = {
    "type": "object",
    "properties": {
        "direct_answer": {"type": "string", "minLength": 1},
        "compliance_level": {"type": "string", "enum": ["Low", "Medium", "High"]},
        "compliance_rationale": {"type": "string", "minLength": 1},
        "key_findings": {"type": "array", "items": {"type": "string"}},
        "disclosure_gaps": {"type": "array", "items": {"type": "string"}},
        "cited_pages": {"type": "array", "items": {"type": "string"}},
    },
    "required": [
        "direct_answer", "compliance_level", "compliance_rationale",
        "key_findings", "disclosure_gaps", "cited_pages",
    ],
    "additionalProperties": False,
}

print("Enforced JSON output schema:")
print(json.dumps(output_schema, indent=2))

print("\n" + "─" * 80)
print("API call configuration:")
print("─" * 80)
print(f"  Model              : gpt-4o (via OPENAI_API_KEY)")
print(f"  Reasoning effort   : medium")
print(f"  Max output tokens  : 4096")
print(f"  Schema mode        : strict (guaranteed valid JSON)")
print(f"  Repair strategy    : If first response fails validation, a repair prompt is sent")

Enforced JSON output schema:
{
  "type": "object",
  "properties": {
    "direct_answer": {
      "type": "string",
      "minLength": 1
    },
    "compliance_level": {
      "type": "string",
      "enum": [
        "Low",
        "Medium",
        "High"
      ]
    },
    "compliance_rationale": {
      "type": "string",
      "minLength": 1
    },
    "key_findings": {
      "type": "array",
      "items": {
        "type": "string"
      }
    },
    "disclosure_gaps": {
      "type": "array",
      "items": {
        "type": "string"
      }
    },
    "cited_pages": {
      "type": "array",
      "items": {
        "type": "string"
      }
    }
  },
  "required": [
    "direct_answer",
    "compliance_level",
    "compliance_rationale",
    "key_findings",
    "disclosure_gaps",
    "cited_pages"
  ],
  "additionalProperties": false
}

────────────────────────────────────────────────────────────────────────────────
API call configuration:
──────────────────────────────────────

## 8. Live Execution — Running Both Conditions

Now we execute the generation for our Aker BP × ESRS S2 example under both conditions.
The `generate_answer()` function in `rag/generation/generator.py` handles caching automatically:
if a cached result exists, it is returned without an API call. Otherwise, the LLM is called
and the result is persisted.

> **Note:** Set `use_cache=True` to use previously cached results (default), or
> `use_cache=False` to force a fresh API call.

In [10]:
# Prepare the evidence passages as plain text (as the pipeline does)
evidence_passages = [ev.text for ev in sample.evidences]

# --- Condition 1: full_system (WITH ESRS context) ---
result_full = generate_answer(
    question=sample.question,
    company=sample.company,
    retrieved_passages=evidence_passages,
    esrs_context=esrs_context,
    condition="full_system",
    use_cache=True,
    question_id=sample.question_id,
    esrs_standard=sample.esrs_standard,
    esrs_disclosure_id=sample.esrs_disclosure_id,
    sample=sample,
)

# --- Condition 2: no_esrs_context (WITHOUT ESRS context) ---
result_no_ctx = generate_answer(
    question=sample.question,
    company=sample.company,
    retrieved_passages=evidence_passages,
    esrs_context="",
    condition="no_esrs_context",
    use_cache=True,
    question_id=sample.question_id,
    esrs_standard=sample.esrs_standard,
    esrs_disclosure_id=sample.esrs_disclosure_id,
    sample=sample,
)

print("✓ Both conditions executed successfully")
print(f"  full_system     → compliance_level: {result_full['compliance_level']}")
print(f"  no_esrs_context → compliance_level: {result_no_ctx['compliance_level']}")
print(f"  Gold standard   → fulfillment_level: {sample.fulfillment_level}")

✓ Both conditions executed successfully
  full_system     → compliance_level: High
  no_esrs_context → compliance_level: High
  Gold standard   → fulfillment_level: High


## 9. Results Comparison — Side by Side

Let's compare the LLM's responses under both conditions for this single example.

### 9.1 Compliance Level and Rationale

In [11]:
def show_result(label, result, gold_level):
    level = result["compliance_level"]
    match = "✓ MATCH" if level == gold_level else "✗ MISMATCH"
    print(f"{'='*80}")
    print(f"  {label}")
    print(f"{'='*80}")
    print(f"  Predicted level : {level}  ({match} with gold: {gold_level})")
    print(f"\n  Rationale:")
    print(textwrap.fill(result["rationale"], width=90, initial_indent="    ", subsequent_indent="    "))
    print()

show_result("Condition: full_system (WITH ESRS context)", result_full, sample.fulfillment_level)
show_result("Condition: no_esrs_context (WITHOUT ESRS context)", result_no_ctx, sample.fulfillment_level)

  Condition: full_system (WITH ESRS context)
  Predicted level : High  (✓ MATCH with gold: High)

  Rationale:
    The retrieved report evidence clearly and substantially describes (1) explicit policy
    commitments covering forced labour, trafficking and child labour (code of conduct and
    human rights policy aligned with UNGPs/ILO/OECD), (2) mandatory supplier requirements
    (supplier declaration and contractual clauses, integrity due diligence), and (3)
    concrete engagement and remediation processes (risk‑based human rights due diligence
    cycle, on‑site audits with worker interviews, corrective action plans, follow‑up, and
    an external integrity channel). These elements directly address ESRS S2-1 (policies)
    and S2-2 (processes for engaging value chain workers) and provide outcome-related
    information (audit findings, follow-up plans). Therefore disclosure fulfillment for
    the question is High.

  Condition: no_esrs_context (WITHOUT ESRS context)
  Predicted l

### 9.2 Key Findings Comparison

In [12]:
print("KEY FINDINGS — full_system (WITH ESRS context)")
print("─" * 60)
for i, f in enumerate(result_full.get("key_findings", []), 1):
    print(f"  {i}. {textwrap.fill(f, width=90, subsequent_indent='     ')}")

print(f"\n{'─'*60}")
print("KEY FINDINGS — no_esrs_context (WITHOUT ESRS context)")
print("─" * 60)
for i, f in enumerate(result_no_ctx.get("key_findings", []), 1):
    print(f"  {i}. {textwrap.fill(f, width=90, subsequent_indent='     ')}")

KEY FINDINGS — full_system (WITH ESRS context)
────────────────────────────────────────────────────────────
  1. Human rights policy and code of conduct explicitly oppose forced/compulsory labour, human
     trafficking and child labour and are aligned with the UN Guiding Principles, ILO
     Declaration and OECD Guidelines (Evidence 6, Evidence 3).
  2. All new suppliers must sign a supplier declaration and contractual clauses; Aker BP
     applies a business partner integrity procedure and integrity due diligence before
     engagement (Evidence 2, Evidence 7, Evidence 11).
  3. Aker BP runs a risk‑based human rights due diligence cycle embedded in governance and
     processes (identification, assessment, action, monitoring, communication,
     grievance/remedy) (Evidence 1, Evidence 4, Evidence 11).
  4. In 2024 Aker BP conducted seven independent on‑site human‑rights audits at 10 production
     sites across seven countries (covering >400 workers) focused explicitly on human
     

### 9.3 Disclosure Gaps Comparison

In [13]:
print("DISCLOSURE GAPS — full_system (WITH ESRS context)")
print("─" * 60)
for i, g in enumerate(result_full.get("disclosure_gaps", []), 1):
    print(f"  {i}. {textwrap.fill(g, width=90, subsequent_indent='     ')}")

print(f"\n{'─'*60}")
print("DISCLOSURE GAPS — no_esrs_context (WITHOUT ESRS context)")
print("─" * 60)
for i, g in enumerate(result_no_ctx.get("disclosure_gaps", []), 1):
    print(f"  {i}. {textwrap.fill(g, width=90, subsequent_indent='     ')}")

DISCLOSURE GAPS — full_system (WITH ESRS context)
────────────────────────────────────────────────────────────
  1. No separate supplier code of conduct is in place (company states this explicitly);
     supplier expectations are delivered via supplier declaration and contractual clauses
     rather than a standalone supplier code (Evidence 2) — ESRS allows supplier code or
     equivalent but presence of a standalone code is absent.
  2. Limited disclosure of engagement frequency/stages and formal assessment of effectiveness:
     while audit activity and some responsible roles are described, the report gives
     limited structured detail on the regularity/frequency of direct engagement with
     value‑chain workers (beyond audits and that frequency varies by contract) and limited
     information on how worker perspectives have changed specific decisions (Evidence 1,
     Evidence 5).
  3. Scope limitation by tiers: most assurance and audits are focused on tier‑1 suppliers (one
    

### 9.4 Quantitative Comparison Summary

In [14]:
comparison_data = {
    "Metric": [
        "Compliance level",
        "Matches gold standard",
        "Answer length (chars)",
        "Rationale length (chars)",
        "# Key findings",
        "# Disclosure gaps",
        "# Cited pages",
    ],
    "full_system": [
        result_full["compliance_level"],
        "✓" if result_full["compliance_level"] == sample.fulfillment_level else "✗",
        f"{len(result_full.get('answer', '')):,}",
        f"{len(result_full.get('rationale', '')):,}",
        str(len(result_full.get("key_findings", []))),
        str(len(result_full.get("disclosure_gaps", []))),
        str(len(result_full.get("cited_pages", []))),
    ],
    "no_esrs_context": [
        result_no_ctx["compliance_level"],
        "✓" if result_no_ctx["compliance_level"] == sample.fulfillment_level else "✗",
        f"{len(result_no_ctx.get('answer', '')):,}",
        f"{len(result_no_ctx.get('rationale', '')):,}",
        str(len(result_no_ctx.get("key_findings", []))),
        str(len(result_no_ctx.get("disclosure_gaps", []))),
        str(len(result_no_ctx.get("cited_pages", []))),
    ],
}

# Display as a simple table
header = f"{'Metric':<30} {'full_system':>20} {'no_esrs_context':>20}"
print(header)
print("─" * len(header))
for i, metric in enumerate(comparison_data["Metric"]):
    print(f"{metric:<30} {comparison_data['full_system'][i]:>20} {comparison_data['no_esrs_context'][i]:>20}")

Metric                                  full_system      no_esrs_context
────────────────────────────────────────────────────────────────────────
Compliance level                               High                 High
Matches gold standard                             ✓                    ✓
Answer length (chars)                         1,656                2,107
Rationale length (chars)                        782                  887
# Key findings                                    7                    9
# Disclosure gaps                                 4                    5
# Cited pages                                     6                    9


## 10. Qualitative Observations

Looking at this single example, several patterns emerge that are representative of the
broader RQ2 findings:

**Compliance level:** Both conditions correctly predict **High** fulfillment, matching the
gold standard. This is consistent with the aggregate finding that ESRS context injection does
not significantly improve level prediction accuracy.

**Rationale style differences:**
- The `full_system` rationale explicitly references ESRS disclosure IDs (e.g., "S2-1", "S2-2")
  when explaining why the compliance level is High. The injected regulatory text gives the LLM
  a framework to map the company's disclosures against.
- The `no_esrs_context` rationale relies entirely on the content of the retrieved evidence to
  make its assessment, using more general compliance reasoning.

**Key findings and gaps:** Both conditions identify similar substantive findings (policies,
audit activity, grievance channels). However, the `no_esrs_context` condition sometimes produces
more detailed factual observations (e.g., specific audit finding counts) since it focuses
entirely on the evidence rather than mapping to ESRS requirements.

This pattern aligns with the aggregate RQ2 result: ESRS context injection does not meaningfully
improve either compliance-level prediction or rationale quality. The evidence retrieved by the
RQ1 pipeline appears to be sufficient for the LLM to reason about disclosure fulfillment without
explicit regulatory guidance.

## 11. Appendix — Full Raw Responses

For completeness, the full JSON responses from both conditions are shown below.

In [15]:
print("=" * 80)
print("FULL RESPONSE — full_system (WITH ESRS context)")
print("=" * 80)
# Show all fields except raw_response (which is the unparsed JSON string)
display_full = {k: v for k, v in result_full.items() if k != "raw_response"}
print(json.dumps(display_full, indent=2, ensure_ascii=False))

FULL RESPONSE — full_system (WITH ESRS context)
{
  "compliance_level": "High",
  "answer": "Aker BP discloses a clear set of policies and engagement processes to identify and address negative impacts on workers in its value chain, including forced and child labour. Key policy elements are: the code of conduct and a human rights policy that explicitly oppose forced/compulsory labour, human trafficking and child labour and state alignment with the UN Guiding Principles on Business and Human Rights, the ILO Declaration and the OECD Guidelines. For suppliers, Aker BP requires a supplier declaration and contractual clauses (all new suppliers must sign), applies an integrity due diligence (IDD) / business partner integrity procedure before contracting, and integrates human-rights considerations into its sourcing and contract management. Engagement and due-diligence processes described include a risk‑based human rights due diligence cycle (identification, assessment, preventive/corrective ac

In [16]:
print("=" * 80)
print("FULL RESPONSE — no_esrs_context (WITHOUT ESRS context)")
print("=" * 80)
display_no_ctx = {k: v for k, v in result_no_ctx.items() if k != "raw_response"}
print(json.dumps(display_no_ctx, indent=2, ensure_ascii=False))

FULL RESPONSE — no_esrs_context (WITHOUT ESRS context)
{
  "compliance_level": "High",
  "answer": "Aker BP discloses a suite of policies and engagement processes to identify and address negative impacts on workers in its value chain, including forced and child labour. Policy instruments: the Code of Conduct and a published Human Rights Policy (aligned with the UN Guiding Principles, ILO Declaration and OECD Guidelines) explicitly oppose forced/compulsory and child labour and apply to suppliers, contractors and partners. Pre-contract and contractual measures: a business partner integrity procedure and an integrity due diligence (IDD) / supplier declaration are required for new suppliers and embedded in contract clauses to communicate expectations and require remediation where needed. Due diligence and monitoring: a risk‑based human rights due diligence process is described (six-step cycle embedding identification, assessment, action, monitoring, communication and grievance/remedy) and 